# VAE classifier for uncertainty estimation.

### Libraries.

In [ ]:
from collections.abc import Callable
from itertools import chain, combinations
import math
import sys
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import energy_distance
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import torch

from sc_exp_design.constants import DataFields, LossFields, ParamsFields
from sc_exp_design.data import DataManager, SequentialDataLoader
from sc_exp_design.networks import ConditionEncoder, MLPGaussianNoiseModel, PerturbationApproximatePosterior
from sc_exp_design.training.utils import compute_pert_inference_loss

from sc_exp_design.training.base import BaseTrainer
from sc_exp_design.training.callbacks import BaseCallBack


sys.path.insert(0, '../../utils')
from fetch_data import get_concatenated_transformed_obs_columns
from vae_utils import log_prob_normal, Dkl_standard_normal

### Constants.

In [ ]:
# working locally?
IS_RUN_LOCALLY = True

# paths
if IS_RUN_LOCALLY:
    H5AD_PATH = "/Users/lorenzo.consoli/Work/data/collab-goettgens/ds_HID01_logicle_100k_UMAP_ann.h5ad"
else:
    H5AD_PATH = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/gdrive/ds_HID01_logicle_100k_UMAP_ann.h5ad"

# obs columns
BATCH_COVARIATES = [
    'Time Stamp',
    'absolute_number_multiplier_[2^n]',
    'cell_counts',
    'count_beads',
    'counts_source_cell_phenotype',
    'cytometer',
    'cytometer_id',
    'cytometer_serial_no',
    'date',
    'days_of_culture',
    'experiment_id',
    'experiment_number',
    'hydrogel_type',
    'plate',
    'replicate',
    'source_cell_phenotype',
    'source_id',
    'total_count_beads',
    'well_id'
]
CLUSTERING_KEYS = ['batch','leiden']
EXPERIMENTAL_COVARIATES = [
    '740-yp_[µm]', 'mtg_[µm]',
    'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]',
    'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]',
    'um729_[µm]', 'scf_[ng_ml]',
    'butyzamide_[nm]', 'retinoic_acid_[µm]',
    'ldl_[ng_ml]', 'il3_[ng_ml]',
    'o2_[%]',
]
PROTOCOL_CONCAT_OBSM_KEY = "protocol_concat"
SCATTER_COLUMNS = ['FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width']
TARGET_COVARIATES = ["cell_type", ]

# Tranformations
LOG1P_EXP_COL = ["um171_[nm]", "um729_[µm]", "scf_[ng_ml]", 
                 "butyzamide_[nm]", "o2_[%]",
                 "sr1_[nm]", "mtg_[µm]", "rhflt3l_[ng_ml]", "gm-csf_[ng_ml]"]
LOG21P_EXP_COL = ["ldl_[ng_ml]", 
    "il3_[ng_ml]", "retinoic_acid_[µm]",
]

# adata keys
X_CHANNEL_OBSM_KEY = "X_channel"
X_SCATTER_OBSM_KEY = "X_scatter"
CONCAT_FEATS_OBSM_KEY_SUFFIX = "scatter_concat"
STANDARDIZED_FEATS_KEY_SUFFIX = "standardized"
CONDITION_CONCAT_OBSM_KEY = "protocol_axes_concat"
X_PCs_OBSM_KEY ='X_pca'
X_UMAP_OBSM_KEY = 'X_umap'

# validation split
VALIDATION_SIZE = 3e-1
RANDOM_SEED = 42

# model training
TRAIN_BATCH_SIZE = 256

# device
if IS_RUN_LOCALLY:
    DEVICE = "mps"
else:
    DEVICE = "cuda"


### Defining utility functions and classes. 

In [ ]:
class VAEClassifier(torch.nn.Module):
    linear_proj_config = {
        "hidden_dims":(),
        "use_dropout": False,
        "use_batchnorm": False,
        "activation_class": torch.nn.Identity,
        "final_activation_class": torch.nn.Identity,
    }

    def __init__(
        self,
        state_dim: int,
        state_latent_dim: int,
        target_output_dims: dict[str, int],
        condition_latent_dim: int | None = 2,
        encoder_mlp_kwargs=None,
        decoder_mlp_kwargs=None,
        layers_before_pooling: dict | None = None,
        layers_after_pooling: dict | None = None,
        is_conditional: bool = True,
        noise_models: dict | None = None,
        covariate_kwargs: dict | None = None,
        pert_cov_estimation_modes: dict | None = None,
    ):
        if is_conditional:
            assert condition_latent_dim is not None
            assert layers_before_pooling is not None
            assert layers_after_pooling is not None

        super().__init__()
        self.state_dim = state_dim
        self.state_latent_dim = state_latent_dim
        self.target_output_dims = target_output_dims
        self.condition_latent_dim = condition_latent_dim
        self.encoder_mlp_kwargs = {} if encoder_mlp_kwargs is None else encoder_mlp_kwargs
        self.decoder_mlp_kwargs = {} if decoder_mlp_kwargs is None else decoder_mlp_kwargs
        self.layers_before_pooling = layers_before_pooling
        self.layers_after_pooling = layers_after_pooling
        self.is_conditional = is_conditional
        self.noise_models = noise_models
        self.covariate_kwargs = covariate_kwargs
        self.pert_cov_estimation_modes = pert_cov_estimation_modes
        self.vae_modules = self._init_modules()

    def _init_modules(self):
        modules = {}
        modules["encoder"] = MLPGaussianNoiseModel(
            self.encoder_input_dim,
            self.state_latent_dim,
            latent_dim=self.state_latent_dim,
            cov_estimation_mode="anisotropic",
            encoder_mlp_kwargs=self.encoder_mlp_kwargs,
            mean_mlp_kwargs=self.linear_proj_config,
            cov_mlp_kwargs=self.linear_proj_config,
        )
        modules["decoder"] =  PerturbationApproximatePosterior(
            self.decoder_input_dim,
            freeze_grads=False,
            target_output_dims=self.target_output_dims,
            noise_models=self.noise_models,
            covariate_kwargs=self.covariate_kwargs,
        )
        if self.is_conditional:
            modules["condition_encoder"] = ConditionEncoder(
                self.condition_latent_dim,
                layers_before_pooling=self.layers_before_pooling,
                layers_after_pooling=self.layers_after_pooling,
            )
        return torch.nn.ModuleDict(modules)

    def get_latent_params(
        self,
        x,
        latent_condition=None,
    ):
        # handling input for encoder
        if self.is_conditional:
            assert latent_condition is not None
            encoder_input = torch.concatenate((x, latent_condition), dim=-1)
        else:
            encoder_input = x
        return self.vae_modules["encoder"](encoder_input)

    def get_decoded_params(
        self,
        latent_states,
        latent_condition=None,
    ):
        # handling input for encoder
        if self.is_conditional:
            assert latent_condition is not None
            decoder_input = torch.concatenate((latent_states, latent_condition), dim=-1)
        else:
            latent_condition = None
            decoder_input = latent_states
        return self.vae_modules["decoder"](decoder_input)

    def encode_condition(
        self,
        condition=None,
    ):
        assert condition is not None
        return self.vae_modules["condition_encoder"](condition)
    
    def sample_latent_states(
        self,
        N,
        x,
        latent_condition=None,
    ):
        # encoding states
        latent_params = self.get_latent_params(x, latent_condition=latent_condition)
        return self.sample_from_params(N, latent_params)
    
    def sample_observations(
        self,
        N,
        latent_states,
        latent_condition=None,
    ):
        # encoding states
        latent_params = self.get_decoded_params(latent_states, latent_condition=latent_condition)
        return self.sample_from_params(N, latent_params)

    def sample_prior_predictive(
        self,
        n_prior_samples,
        n_likelihood_samples,
        condition=None,
    ):
        # encoding conditions
        if self.is_conditional:
            assert condition is not None
            cond_dict = {
                k: v.type(torch.float32).to(self.device)  for k, v in condition.items()
            }
            latent_cond = self.encode_condition(cond_dict)
            latent_cond = latent_cond.unsqueeze(0)
            # print(latent_cond.shape)
        else:
            latent_cond = None
        z_prior = torch.randn((n_prior_samples, self.vae_modules["encoder"].output_dim), device=self.device, dtype=torch.float32)
        # print(z_prior.shape)
        if latent_cond is not None:
            latent_cond = latent_cond.unsqueeze(0).repeat(n_likelihood_samples, *(1 for _ in latent_cond.shape))
        x_params = self.get_decoded_params(z_prior, latent_condition=latent_cond)
        # for k, v in x_params.items():print(k, v.shape)
        x_samples = self.sample_from_params(n_likelihood_samples, x_params)
        return {
            "z_samples": z_prior,
            "x_params": x_params,
            "x_samples": x_samples,
            "latent_cond": latent_cond
        }

    def sample_posterior_predictive(
        self,
        x,
        n_posterior_samples,
        n_likelihood_samples,
        condition=None,
    ): 
        # encoding conditions 
        if self.is_conditional:
            assert condition is not None
            cond_dict = { k: v for k, v in condition.items() }
            latent_cond = self.encode_condition(cond_dict)
        else: latent_cond = None
        z_params = self.get_latent_params(x, latent_condition=latent_cond)
        z_posterior = self.sample_from_params(n_posterior_samples, z_params) # m: (N, B, Z)
        if latent_cond is not None:
            latent_cond = latent_cond.unsqueeze(0).repeat(n_posterior_samples, *(1 for _ in latent_cond.shape))
        x_params = self.get_decoded_params(z_posterior, latent_condition=latent_cond)
        x_samples = self.sample_from_params(n_likelihood_samples, x_params)
        return { "z_params": z_params, "z_samples": z_posterior, "x_params": x_params, "x_samples": x_samples, "latent_cond": latent_cond }

    @property
    def encoder_input_dim(
        self,
    ):
        if self.is_conditional:
            return self.state_dim + self.condition_latent_dim
        return self.state_dim

    @property
    def decoder_input_dim(
        self,
    ):
        if self.is_conditional:
            return self.state_latent_dim + self.condition_latent_dim
        return self.state_latent_dim

    @staticmethod
    def sample_from_params(N, params):
        # parsing parameter dictionary
        mean = params[ParamsFields.MEAN]
        std = params[ParamsFields.COVARIANCE]

        # sampling noise and matching shapes
        noise = torch.randn((N, *mean.shape), device=mean.device, dtype=torch.float)
        mean = mean.unsqueeze(0)
        cov = torch.exp(2*std.unsqueeze(0))
        return mean + torch.sqrt(cov)*noise


class VAEClassifierTrainer(BaseTrainer):
    def __init__(
        self,
        model,
        optimizer: torch.optim.Optimizer,
        lr_scheduler: torch.optim.lr_scheduler.LRScheduler | None = None,
        lr_scheduler_step: Literal["grad_step", "valid_step"] = "grad_step",
        callbacks: BaseCallBack | None = None,
        grad_steps_log_interval: int | None = 1,
        num_grad_accumulation_steps: int = 100,
        reg_strength = 1e-0,
        n_mc_samples = 100,
        cond_reg_mode = None,
        cond_reg_strength=1e-1,
        eps=1e-15,
        loss_fn_kwargs={},
    ):
        self.model = model
        self.optimizer = optimizer
        self.lr_scheduler = lr_scheduler
        self.lr_scheduler_step = lr_scheduler_step
        self.callbacks = callbacks
        self.grad_steps_log_interval = grad_steps_log_interval
        self.num_grad_accumulation_steps = num_grad_accumulation_steps 
        self.reg_strength = reg_strength
        self.n_mc_samples = n_mc_samples
        self.cond_reg_mode = cond_reg_mode
        self.cond_reg_strength = cond_reg_strength
        self.eps = eps
        self.loss_fn_kwargs = loss_fn_kwargs

    def _train_step(
        self,
        step_idx: int,
        batch: dict[str, torch.Tensor],
    ):
        # parsing batch
        x = batch[DataFields.STATE_DATA] # (B, D)
        condition = batch.get(DataFields.PERTURBATION_DATA, None) # (B, C)
        targets = {k: v.unsqueeze(0).repeat(self.n_mc_samples, *(1 for _ in v.shape)) for k, v in batch[DataFields.TARGET_CATEGORIES].items()}

        # encoding condition
        if condition is not None and self.model.is_conditional:
            latent_cond = self.model.encode_condition(condition)
        else:
            latent_cond = None
        
        # encoder and latent regularization
        q_params = self.model.get_latent_params(x, latent_condition=latent_cond) # m: (B, Z), s_i: (B, 1), s_a: (B, Z)
        z_samples = self.model.sample_from_params(self.n_mc_samples, q_params) # m: (N, B, Z)
        dkl = Dkl_standard_normal(q_params, eps=self.eps) # ()
        reg_error = -0.5*self.reg_strength*dkl

        # print(torch.exp(2*q_params[ParamsFields.COVARIANCE]))
        assert len(reg_error.shape) == 0, len(reg_error.shape)
        assert torch.all(torch.isfinite(q_params[ParamsFields.MEAN]))
        assert torch.all(torch.isfinite(q_params[ParamsFields.COVARIANCE]))
        assert torch.all(q_params[ParamsFields.COVARIANCE] + self.eps != 0)
        assert torch.isfinite(reg_error)
        assert torch.all(torch.isfinite(z_samples))
    
        # optional condition regularization
        if self.cond_reg_mode is None or latent_cond is None:
            cond_reg = 0.0
        elif self.cond_reg_mode == "l1":
            cond_reg = torch.mean(
                torch.sum(
                    torch.abs(latent_cond),
                    dim=-1
                )
            )
        elif self.cond_reg_mode == "l2":
            cond_reg =  torch.mean(
                torch.sqrt(
                    torch.sum(
                        torch.square(latent_cond),
                        dim=-1
                    )
                )
            )
        cond_rec_err = self.cond_reg_strength*cond_reg
        if isinstance(cond_reg, torch.Tensor):
            cond_reg = cond_reg.item()

        # decoder and reconstruction
        if latent_cond is not None:
            latent_cond=latent_cond.unsqueeze(0).repeat(self.n_mc_samples, *[1 for _ in z_samples.shape[1:]])
        p_params = self.model.get_decoded_params(
            z_samples,
            latent_condition=latent_cond 
        )
        for k, v in p_params.items():
            p_params[k] = v.reshape(-1, v.shape[-1])        
        for k, v in targets.items():
            # targets[k] = v.reshape(-1, v.shape[-1])
            targets[k] = v.reshape(-1)

        # computing loss
        rec_error, _ = compute_pert_inference_loss(
            p_params,
            targets,
            self.model.vae_modules["decoder"].noise_models,
            pert_cov_estimation_modes=self.model.pert_cov_estimation_modes, # TODO: we dont need it for the moment but we will need to pass it at some point.
            allow_noise_model_to_be_none=True,
            loss_fn_kwargs=self.loss_fn_kwargs
        )

        # print(rec_error, reg_error, cond_rec_err)
        assert len(rec_error.shape) == 0, len(rec_error.shape)
        assert torch.all(torch.isfinite(z_samples))
        assert torch.all(torch.isfinite(x))
        assert torch.isfinite(rec_error)

        # aggregating loss and returning it
        loss = rec_error + reg_error + cond_rec_err
        return loss, {LossFields.LOSS: loss.item(), "dkl": -dkl.item(), "rec_error": rec_error.item(), "cond_reg": cond_reg, "grad_step": step_idx}

    def _validation_step(
        self,
        batch: dict[str, torch.Tensor],
    ) -> tuple[torch.Tensor]:
        """"""
        # predictions, target = ..., ...
        # return predictions, target
        raise NotImplementedError



## Data.

In [ ]:
adata = sc.read_h5ad(H5AD_PATH)
adata

### Defining tranformations for each experimental column.


In [ ]:
col2transf = {}
for col in EXPERIMENTAL_COVARIATES:
    if col in LOG1P_EXP_COL:
        col2transf[col] = np.log1p
    elif col in LOG21P_EXP_COL:
        col2transf[col] = lambda x: np.log2(x + 1)
    else:
        col2transf[col] = None
print(col2transf)

### Annotating the data with utility function.

In [ ]:
adata = get_concatenated_transformed_obs_columns(adata, col2transf, obsm_col=PROTOCOL_CONCAT_OBSM_KEY)
adata

### Splitting train and validation data.


In [ ]:
train_adata, val_adata = train_test_split(adata, test_size=VALIDATION_SIZE, random_state=RANDOM_SEED)
print("train", train_adata)
print("val", val_adata)

### Normalizing scatter features.

In [ ]:
# retrieving scatter features and optionally normalizing them
X_scatter_train = train_adata.obs[SCATTER_COLUMNS].values
X_scatter_val =  val_adata.obs[SCATTER_COLUMNS].values
X_scatter_mean = X_scatter_train.mean(0)
X_scatter_std = X_scatter_train.std(0)
print("Standardizing scatter features")
X_scatter_train_std = (X_scatter_train - X_scatter_mean)/X_scatter_std
X_scatter_val_std = (X_scatter_val - X_scatter_mean)/X_scatter_std
assert np.isfinite(X_scatter_train_std).all()
assert np.isfinite(X_scatter_val_std).all()
train_adata.obsm[X_SCATTER_OBSM_KEY] = X_scatter_train
val_adata.obsm[X_SCATTER_OBSM_KEY] = X_scatter_val
train_adata.obsm[f"{X_SCATTER_OBSM_KEY}_{STANDARDIZED_FEATS_KEY_SUFFIX}"] = X_scatter_train_std
val_adata.obsm[f"{X_SCATTER_OBSM_KEY}_{STANDARDIZED_FEATS_KEY_SUFFIX}"] = X_scatter_val_std


### Normalizing channel features.

In [ ]:
# concatenating with channel features
train_adata.obsm[X_CHANNEL_OBSM_KEY] = train_adata.X
val_adata.obsm[X_CHANNEL_OBSM_KEY] = val_adata.X
X_channel_train = train_adata.X
X_channel_val = val_adata.X
X_channel_mean = X_channel_train.mean(0)
X_channel_std = X_channel_train.std(0)
print("Standardizing channel features")
X_channel_train_std = (X_channel_train - X_channel_mean)/X_channel_std
X_channel_val_std = (X_channel_val - X_channel_mean)/X_channel_std
assert np.isfinite(X_channel_train).all()
assert np.isfinite(X_channel_val).all()
train_adata.obsm[f"{X_CHANNEL_OBSM_KEY}_{STANDARDIZED_FEATS_KEY_SUFFIX}"] = X_channel_train_std
val_adata.obsm[f"{X_CHANNEL_OBSM_KEY}_{STANDARDIZED_FEATS_KEY_SUFFIX}"] = X_channel_val_std
train_adata, val_adata

### Concatenating channel and scatter features.

In [ ]:
train_adata.obsm[f"{X_CHANNEL_OBSM_KEY}_{STANDARDIZED_FEATS_KEY_SUFFIX}_{CONCAT_FEATS_OBSM_KEY_SUFFIX}"] = np.concatenate((X_channel_train_std, X_scatter_train_std), axis=1)
val_adata.obsm[f"{X_CHANNEL_OBSM_KEY}_{STANDARDIZED_FEATS_KEY_SUFFIX}_{CONCAT_FEATS_OBSM_KEY_SUFFIX}"] = np.concatenate((X_channel_val_std, X_scatter_val_std), axis=1)
train_adata.obsm[f"{X_CHANNEL_OBSM_KEY}_{CONCAT_FEATS_OBSM_KEY_SUFFIX}"] = np.concatenate((X_channel_train, X_scatter_train), axis=1)
val_adata.obsm[f"{X_CHANNEL_OBSM_KEY}_{CONCAT_FEATS_OBSM_KEY_SUFFIX}"] = np.concatenate((X_channel_val, X_scatter_val), axis=1)
train_adata, val_adata

### Defining data manager.

In [ ]:
repr = f"{X_CHANNEL_OBSM_KEY}_{STANDARDIZED_FEATS_KEY_SUFFIX}_{CONCAT_FEATS_OBSM_KEY_SUFFIX}"
dm = DataManager(
    train_adata,
    sample_rep=repr,
    perturbations=(PROTOCOL_CONCAT_OBSM_KEY, ),
    perturbations_in_obsm=(PROTOCOL_CONCAT_OBSM_KEY, ),
    perturbation_reps={PROTOCOL_CONCAT_OBSM_KEY:PROTOCOL_CONCAT_OBSM_KEY},
    load_target_covariates=True,
    target_covariates={"cell_type": "label"}
)

### Retrieving data.

In [ ]:
train_data = dm.get_data(train_adata)
val_data = dm.get_data(val_adata)
perturbation_covariates = train_data.data.perturbation_covariates[0]
print(perturbation_covariates)

### Defining dataloader.

In [ ]:
train_dl = SequentialDataLoader(
    train_data,
    batch_size= TRAIN_BATCH_SIZE,
    device_id=DEVICE,
)


## VAE.

### NN configurations.

In [ ]:
get_mlp_config = lambda hidden_dims: {
    "hidden_dims": hidden_dims,
    "use_batchnorm": False,
    "use_dropout": False,
    "activation_class": torch.nn.ReLU,
    "final_activation_class": torch.nn.Identity,
}
get_lm_config = lambda: get_mlp_config(())

get_cond_encoder_config = lambda condition_input_dim, condition_latent_dim, hidden_dims: {
    "layers_before_pooling": {
        perturbation_covariates: 
        {
            "input_dim": condition_input_dim,
            "output_dim": condition_latent_dim,
            **get_mlp_config(
                hidden_dims
            )
        }
    },
    "layers_after_pooling": {
        "input_dim": condition_latent_dim,
        "output_dim": condition_latent_dim,
        **get_lm_config()
    }
}

get_covariate_kwargs = lambda hidden_dims: {
    "cell_type":{
        "hidden_dims": hidden_dims,
        "activation_class": torch.nn.ReLU,
        "use_dropout": False,
        "use_batchnorm": False,
        "final_activation_class": torch.nn.Identity,
    }
}
pert_cov_estimation_modes = {"cell_type": None}

target_output_dims={"cell_type": adata.obs.cell_type.nunique()}
noise_models={"cell_type": None}

### Initializing vae.

In [ ]:
state_latent_dim = 8
condition_latent_dim = 4
encoder_hidden_dims = (256, 256)
decoder_hidden_dims = (256, 256)
condition_encoder_hidden_dims = (128, )
condition_input_dim = len(EXPERIMENTAL_COVARIATES)
vae = VAEClassifier(
    state_dim=train_data.state_data.shape[-1],
    state_latent_dim=state_latent_dim,
    condition_latent_dim=condition_latent_dim,
    encoder_mlp_kwargs=get_mlp_config(encoder_hidden_dims),
    **get_cond_encoder_config(
        condition_input_dim,
        condition_latent_dim,
        condition_encoder_hidden_dims,
    ),
    # is_conditional=True,
    is_conditional=False,
    target_output_dims=target_output_dims,
    noise_models=noise_models,
    covariate_kwargs=get_covariate_kwargs(decoder_hidden_dims),
    pert_cov_estimation_modes=pert_cov_estimation_modes,
)
vae.to(DEVICE)
vae

### Check parameters.

In [ ]:
# Your custom parameters (with names)
print("Custom parameters:")
for name, param in chain.from_iterable(
    [(f"{module_name}.{p_name}", p) 
     for p_name, p in module.named_parameters()]
    for module_name, module in vae.vae_modules.items()
):
    print(f"{name}: {param.shape}")

# Default PyTorch named_parameters
print("\nDefault named parameters (from vae_modules):")
for name, param in vae.vae_modules.named_parameters():
    print(f"{name}: {param.shape}")


### Training.

In [ ]:
Y_train = train_data.target_data["cell_type"]
lr = 5e-4
num_grad_steps = 10_000

class_weights = torch.from_numpy(compute_class_weight("balanced", classes=np.unique(Y_train), y=Y_train)).float().to(DEVICE)
print(class_weights)

optimizer = torch.optim.Adam(vae.parameters(), lr=lr)
trainer = VAEClassifierTrainer(
    vae,
    optimizer=optimizer,
    loss_fn_kwargs={"weight":class_weights},
    reg_strength=1e-0,
    n_mc_samples=1500,
)
trainer.fit(
    num_grad_steps,
    train_dl,
)

### Plotting training logs.

In [ ]:
fig, axes = trainer.plot_training_logs(
    keys_to_plot=trainer.training_logs.keys(),
    figsize=(15, 3)
)
for ax in axes:
    ax.grid(True)
fig.show()